# 05 · Multi-agent orchestration — every pattern
### agents-as-tools · sequential graph · conditional + parallel graph · Swarm · A2A

One agent juggling flights, hotels, *and* policy gets mediocre at all three. Specialists do better — but now you need to **coordinate** them. There's a spectrum, from "you control the structure" to "the agents figure it out":

```mermaid
flowchart LR
    H[Agents-as-tools<br/>orchestrator calls specialists] --> G[Graph<br/>you define the DAG<br/>order / conditions / parallel]
    G --> S[Swarm<br/>agents hand off to each other]
    S --> A[A2A protocol<br/>agents in separate processes/runtimes]
```

This notebook builds **all** of them on the same TravelMind specialists, so you can feel the trade-offs. Rule of thumb: **use the most constrained pattern that does the job.** Determinism is easier to debug than emergence.

> Prereqs: notebook 00. The orchestration cells run with model access (specialists use mock tools). A2A needs the `[a2a]` extra and spans processes, so its serve/call step is shown as the real commands rather than executed in one cell.


## Shared specialists (built once, reused by every pattern)

Three focused agents, each with a `name` (so others can address it) and a narrow brief. Tools are mock so the logic runs offline of real APIs.


In [ ]:
import os
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
MODEL_HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_SONNET = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

from strands import Agent, tool

@tool
def flight_status(flight_number: str) -> dict:
    """Current status of a flight by its number (e.g. BA117)."""
    return {"BA117": {"status": "Delayed", "delay_min": 180},
            "AA100": {"status": "On time", "delay_min": 0}}.get(flight_number.upper(), {"status": "Unknown"})

@tool
def find_alt_flight(origin: str, dest: str) -> list:
    """Find alternative flights between two airport codes."""
    return [{"flight": "BA179", "from": origin, "to": dest, "price_gbp": 420}]

@tool
def find_hotel(city: str, nights: int) -> list:
    """Find hotels in a city for a number of nights."""
    return [{"name": "Airport Inn", "city": city, "per_night_gbp": 150, "nights": nights}]

flight_agent = Agent(model=MODEL_HAIKU, name="flight_agent",
    tools=[flight_status, find_alt_flight],
    system_prompt="You handle flights only: status checks and finding alternatives. Be concise.")

hotel_agent = Agent(model=MODEL_HAIKU, name="hotel_agent",
    tools=[find_hotel],
    system_prompt="You handle hotels only: find and recommend stays. Be concise.")

policy_agent = Agent(model=MODEL_HAIKU, name="policy_agent",
    system_prompt="You explain airline compensation/rebooking rules (e.g. EU261) plainly and briefly.")

print("specialists ready:", flight_agent.name, hotel_agent.name, policy_agent.name)

## Pattern 1 — Agents-as-tools (hierarchical / orchestrator)

The most controlled pattern. You wrap each specialist in a `@tool`, and a single **orchestrator** agent calls them like any other tool. The orchestrator (use a stronger model here) decides who to involve and synthesizes the final answer.

**Why start here:** it's just tool-calling you already understand, the control flow is explicit, and it composes cleanly.


In [ ]:
@tool
def ask_flight_team(query: str) -> str:
    """Delegate a flight question (status or alternatives) to the flight specialist."""
    return str(flight_agent(query))

@tool
def ask_hotel_team(query: str) -> str:
    """Delegate a hotel question to the hotel specialist."""
    return str(hotel_agent(query))

@tool
def ask_policy_team(query: str) -> str:
    """Delegate a compensation/policy question to the policy specialist."""
    return str(policy_agent(query))

orchestrator = Agent(
    model=MODEL_SONNET,                       # the coordinator benefits from stronger reasoning
    tools=[ask_flight_team, ask_hotel_team, ask_policy_team],
    system_prompt=("You are TravelMind's coordinator. Split the request into parts, "
                   "delegate each to the right specialist tool, then merge into one clear answer."),
)

print(str(orchestrator(
    "My flight BA117 is delayed 3 hours. Am I owed compensation, can you find an alternative "
    "LHR->JFK, and a hotel near JFK for 1 night?"
)))

## Pattern 2 — Graph: a deterministic pipeline

When the steps have a **fixed order**, a `Graph` makes it explicit and inspectable. `GraphBuilder` wires nodes and edges; calling the built graph runs them in dependency order and hands each node's output downstream.

Here: rebook the flight, *then* find a hotel.


In [ ]:
from strands.multiagent import GraphBuilder

b = GraphBuilder()
b.add_node(flight_agent, "flights")
b.add_node(hotel_agent, "hotels")
b.add_edge("flights", "hotels")          # flights runs first, its output flows to hotels
b.set_entry_point("flights")
pipeline = b.build()

result = pipeline("Rebook BA117 from LHR to JFK, then find a hotel near JFK for 1 night.")
print("status:", result.status)
for node_id in result.execution_order:                    # the order nodes actually ran
    print(f"\n[{node_id}]\n", str(result.results[node_id].result)[:240])

## Pattern 3 — Graph with conditional routing + parallel branches

Real workflows branch. A **classifier** node decides the path; **conditional edges** route based on what a prior node produced; multiple edges from one node run **in parallel**.

- `add_edge(from, to, condition=fn)` — the edge only fires when `fn(state)` is `True`.
- The `condition` reads `GraphState.results[node_id].result` — i.e. what an upstream node returned.
- Two edges out of the classifier with the same condition = those nodes run **concurrently**.

Flow: classify → if **disruption**, run *flights* and *policy* in parallel; if **planning**, run *hotels*.


In [ ]:
classifier = Agent(model=MODEL_HAIKU, name="classifier",
    system_prompt="Classify the user's travel request as exactly one word: DISRUPTION or PLANNING.")

def _classified(state, label: str) -> bool:
    nr = state.results.get("classifier")            # the classifier node's result
    return label in (str(nr.result).lower() if nr else "")

def is_disruption(state) -> bool:
    return _classified(state, "disruption")

def is_planning(state) -> bool:
    return _classified(state, "planning")

b = GraphBuilder()
b.add_node(classifier, "classifier")
b.add_node(flight_agent, "flights")
b.add_node(policy_agent, "policy")
b.add_node(hotel_agent, "hotels")
b.set_entry_point("classifier")

# disruption path -> flights AND policy fire in parallel (two conditional edges)
b.add_edge("classifier", "flights", condition=is_disruption)
b.add_edge("classifier", "policy",  condition=is_disruption)
# planning path -> hotels
b.add_edge("classifier", "hotels",  condition=is_planning)

router = b.build()
result = router("My flight BA117 just got cancelled — what are my options and rights?")
print("status:", result.status)
print("ran:", result.execution_order)                 # expect classifier, then flights + policy
for node_id in result.execution_order:
    print(f"\n[{node_id}]\n", str(result.results[node_id].result)[:200])

## Pattern 4 — Swarm: autonomous handoff (no central controller)

In a **Swarm**, there's no orchestrator. Each agent can **hand off** to another when the task moves outside its lane. Strands auto-injects a `handoff_to_agent(agent_name, message, context=None)` tool into every swarm member — so the agents route *themselves*.

You set an `entry_point` and guard rails (`max_handoffs`, `max_iterations`) so it can't loop forever.

**When to use:** open-ended tasks where the right sequence isn't known up front. The cost: less predictable paths — harder to debug than a Graph.


In [ ]:
from strands.multiagent import Swarm

triage = Agent(model=MODEL_HAIKU, name="triage",
    system_prompt=("You triage travel requests, then hand off: flights -> flight_agent, "
                   "hotels -> hotel_agent, compensation/policy -> policy_agent. "
                   "Hand off using the handoff tool; don't answer specialist questions yourself."))

swarm = Swarm(
    [triage, flight_agent, hotel_agent, policy_agent],
    entry_point=triage,
    max_handoffs=6,
    max_iterations=10,
)

result = swarm("My flight BA117 is delayed 3 hours — what are my rights and what are my options?")
print("status:", result.status)
print("path :", [getattr(n, "node_id", n) for n in result.node_history])   # who handled it, in order
for node_id, nr in result.results.items():
    print(f"\n[{node_id}]\n", str(nr.result)[:200])

## Pattern 5 — A2A protocol: agents in different processes/runtimes

The previous patterns run agents **in one process**. When specialists are owned by different teams, written in different frameworks, or deployed as separate Runtime services, they talk over **A2A** (Agent-to-Agent) — an open protocol. Each agent publishes an **agent card** (its capabilities) at a URL; clients discover and call it.

Install the extra:
```bash
pip install 'strands-agents[a2a]'
```


In [ ]:
from strands.multiagent.a2a import A2AServer

# Wrap any Strands agent as an A2A server. In real use this runs as its OWN process
# (or a Runtime deployed with protocol='A2A'), not inline in a notebook cell.
flight_server = A2AServer(flight_agent, host="0.0.0.0", port=9000)

# What other agents discover about this one (its capabilities / skills):
print("agent card URL:", flight_server.agent_card_url)
# flight_server.serve()        # <-- blocks and serves; run in a dedicated process

**Calling an A2A agent.** A client agent uses the `a2a_client` tool (from `strands_tools`) or the A2A SDK to fetch the card and invoke skills — so an orchestrator can treat a *remote* agent exactly like a local tool. On AgentCore, deploy the server side with `Runtime().configure(protocol="A2A")` (notebook 02) and it becomes a networked, autoscaled A2A endpoint.

```python
# client side sketch (runs against a serving A2A endpoint):
# from strands import Agent
# from strands_tools import a2a_client
# client = Agent(model=MODEL_HAIKU, tools=[a2a_client])
# client("Use the agent at http://localhost:9000 to check flight BA117.")
```


## Handing **data** between agents — Memory branches

Orchestration moves *control*; sometimes you also need to hand *structured context* cleanly. From notebook 03: a memory **branch** forks a conversation, so a side-investigation (say, a rebooking thread) has its own lineage and another agent can pick it up as handed-over context.

```python
mc.create_event(memory_id=MEMORY_ID, actor_id=ACTOR, session_id=SESSION,
                branch={"rootEventId": <event_id>, "name": "rebooking"},
                messages=[("BA117 cancelled; alternatives needed.", "USER")])
# the rebooking agent reads this branch as its starting context
```


## Choosing a pattern

| Pattern | Control | Parallel? | Best for | Watch out |
|---|---|---|---|---|
| Agents-as-tools | High | Orchestrator-driven | Clear delegation, synthesis | Orchestrator is a bottleneck/single point |
| Graph (sequential) | Highest | No | Fixed pipelines | Rigid — every path pre-defined |
| Graph (conditional/parallel) | High | Yes | Branching workflows, fan-out/in | Conditions must be exhaustive |
| Swarm | Low | Emergent | Open-ended tasks | Less predictable; cap handoffs |
| A2A | Distributed | Across services | Cross-team / cross-runtime agents | Network, auth, discovery overhead |

**Skeptic's corner.** Multi-agent is not automatically better. Every extra agent adds latency, tokens, and failure modes. Start with one good agent; split into specialists only when a single agent's quality visibly drops or teams/ownership demand it. When you do split, pick the **least** autonomous pattern that works.

## Next

**`06_travelmind_capstone.ipynb`** — assemble everything: a TravelMind flight-disruption agent that uses Memory (traveler prefs), tools (fare math + flight API), a multi-agent core, runs on Runtime, and is observable. Local → deployed → invoked.
